# 4b. PUSH transfer (DSP 2025-1, eunomia RPC)

In **PUSH** mode the Consumer names a sink in its request and the Provider delivers the data to
it: on start the Provider subscribes to its backend through the PUSH connector, and the backend's
events flow through the Provider's dataplane to the sink. The connectors of this notebook use no
authentication (`setup_catalog_policies.ipynb`).

Lifecycle walked here over `/dsp/current/transfers/rpc/*` and `/api/v1/transfer-agent/*`:
`REQUESTED` -> `STARTED` (pushing an event through the dataplane to the Consumer's sink) ->
`SUSPENDED` -> `STARTED` -> `COMPLETED`, plus the role refusal of a start sent by the Consumer.

**Runs on its own.** Prerequisites: `setup_catalog_policies.ipynb` and `contract_negotiation.ipynb`,
both guarded. The demo backend (`127.0.0.1:8082`) and sink (`127.0.0.1:1112`) are started
automatically in the kernel if not already running.

## 1. Prerequisites

Two guards make the notebook runnable on its own and cheap to rerun in the same kernel:
- the catalog notebook runs unless the kernel already holds its variant (`catalog_setup == "no_auth"`);
  it onboards the participants first if needed;
- the negotiation runs unless the kernel already holds an agreement from `contract_negotiation.ipynb` on the loaded
  dataset (`agreement_source == ("rpc", target_id)`).

In [1]:
if globals().get("catalog_setup") != "no_auth":
    %run ./setup_catalog_policies.ipynb
if globals().get("agreement_source") != ("rpc", target_id):
    %run ./contract_negotiation.ipynb
ensure_demo_servers()
print(f"Transferring under agreement {agreement_id}")

Authority URL: http://127.0.0.1:1500
Consumer URL:  http://127.0.0.1:1100
Provider URL:  http://127.0.0.1:1200
Wallet linked for Authority (http://127.0.0.1:1500): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Consumer (http://127.0.0.1:1100): wallet id=fafnir-local, name=fafnir-wallet
Wallet linked for Provider (http://127.0.0.1:1200): wallet id=fafnir-local, name=fafnir-wallet
HTTP 200 GET http://127.0.0.1:1100/api/v1/mates/myself
{
  "participant_id": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ5cS12NXdJNFM2MTl2SnJITmpENDVra0owUGw4VGlBTzZYTnFJc1o4SDBaTEs5T2JoWTBTeGtEQ2JlZWd2RlZEU0MzM182N3lVZVpUWFI2SjFXUExHdHNtLU51SXJtR1RVRVgteFA0WnFLVGRFUTFSdUV0MExtMUhtdkdJWG1zLVJrTXRNVjcxRWt0MjNiZFYzX01WTjRUOUd5WnlaQTB6RDRlTFUxRTRTRzlRMGl4MGFRbC16eGdsVFRYQ2l4VTdCTzRXcHRkaUp3a0Jid2hMbnVqeXFzSmNvQXpPU000aXZtZlRTVW1Yd0Nrc1gySTFxVTNnVWdtUGFtbmEta3ZUb3FYZUk4ODNlRTFkSWNuUG53dmJlRmpfa0xuQ2JrNzhjNkVkRXZ0OVBMa3ByTWZfdmVxeVlOWjB0WlpYUGM5c0lRVlMyUG5xZkN5RUhsWWVuMXNXclEifQ",
  "participant_n

### How the transfer RPCs and Management API work

As in the negotiation, each participant asks its own agent to send the next DSP message, through
`/dsp/current/transfers/rpc/setup-{request,start,suspension,completion,termination}`. Bodies are
eunomia JSON-LD (`@context` the eunomia context, `@type` `RpcTransfer<Kind>Message`) and the
answer is an `RpcTransferResult`: `request`, the peer's `response`, and `transferAgentModel`, the
caller's own `TransferProcessView` (`id`, `state`, `stateMetadata`, `correlation` with both pids).

The Transfer Agent and Dataplane Management APIs under `/api/v1/transfer-agent/` expose:
- `/transfer-processes/{id}`: the control-plane transfer record.
- `/transfer-messages/process/{id}`: every DSP transfer message exchanged and its state transition.
- `/dataplane/dataplane-processes/transfer-process/{id}`: the dataplane process record (`ingressConfig`, `egressConfig`, `flowControl`).
- `/dataplane/dataplane-processes/{dataplaneId}/{logs,events}`: transition logs and proxied event records.

In [2]:
def transfer_step(response, expected_state):
    """Prints an RpcTransferResult and checks the caller's process reached expected_state."""
    answer = show(response)
    process = answer["transferAgentModel"]
    assert process["state"] == expected_state, f"expected {expected_state}, got {process['state']}"
    print(f"\n==> {process['state']}")
    return process

## 2. Transfer request with a sink (`rpc/setup-request`, Consumer -> Provider)

The Consumer requests the `http+push` distribution and names, in `dataAddress`, the sink where
the Provider must deliver the data, with the credentials the Provider must present there.

In [3]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferRequestMessage",
    "associatedAgentPeer": data_space_provider_participant_id,
    "providerAddress": data_space_provider_dsp_endpoint,
    "callbackAddress": data_space_consumer_dsp_endpoint,
    "agreementId": agreement_id,
    "format": "http+push",
    "dataAddress": {
        "endpointType": "https://w3id.org/idsa/v4.1/HTTP",
        "endpoint": "http://127.0.0.1:1112",
        "endpointProperties": [
            {"name": "authorization", "value": "TOKEN-ABCDEFG"},
            {"name": "authType", "value": "bearer"},
        ],
    },
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-request", headers=json_header, json=payload)
consumer_transfer = transfer_step(response, "REQUESTED")
transfer_consumer_pid = consumer_transfer["correlation"]["consumerPid"]
transfer_provider_pid = consumer_transfer["correlation"]["providerPid"]

def own_transfer_id(agent_url, provider_pid):
    """The id an agent gives the transfer whose providerPid is `provider_pid`."""
    processes = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-processes", headers=json_header).json()
    return next(p["id"] for p in processes["items"] if p["correlation"]["providerPid"] == provider_pid)

# RPC calls name the transfer by each agent's own process id, never by DSP pids.
transfer_consumer_process_id = consumer_transfer["id"]
transfer_provider_process_id = own_transfer_id(data_space_provider, transfer_provider_pid)
print(f"consumer process: {transfer_consumer_process_id}\nprovider process: {transfer_provider_process_id}")

HTTP 201 POST http://127.0.0.1:1100/dsp/current/transfers/rpc/setup-request
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferRequestMessage",
    "agreementId": "urn:agreement:cac82465-0533-4aba-8f64-02d10ac9cd09",
    "associatedAgentPeer": "did:jwk:eyJlIjoiQVFBQiIsImt0eSI6IlJTQSIsIm4iOiJ2UFV3bEVyS09BQnR4eW1TNFRGeV9jd2YxczgxR3VNSXNlMlNoS1NkdHgyXzZEamh0X2JrZHVIX3AzY1VraVNzbTdQdFRyZ184MTVRUHNid1dENkJmbWk4OVhvV21ZTUhqSXI2Q1VPdUk2M2JSY09FakVmMnpvWEJockMtR2ViRS1qSk5zNTFQWTZ0MUYwWFIzMTVLczlqLWdHSzZ1MGxTSXMySkdXWFBxVFV2QnlTVzhpeE5MNzBjaHM4OXItZnRvZDRDakg2Mnlod2IzYk1WeWxrTjNIYnZWQVlkRXRpM1FnUXlwRFRNMTViSlVvUy1hZlZRUGRIMlYxbXA3X2FJY2FEUEhTVG1xMUhZQTgxc01lZm9mNTdfRFJWQUxrcXBSTzVEa2hSOTFfeEZqQXRCODEtSnRySHN2c19HZ0lvN24yUnh6MGZScnV4VkNQWlBtVVRDMncifQ",
    "callbackAddress": "http://127.0.0.1:1100/dsp/current",
    "dataAddress": {
      "endpoint": "h

## 3. Start (`rpc/setup-start`, Provider -> Consumer)

The Provider subscribes to its backend (`127.0.0.1:8082/subscriptions/subscribe`) through the PUSH
connector and transitions the transfer and dataplane to `STARTED`.

In [4]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "@id": transfer_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-start
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
    "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
      "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb",
      "agree

## 4. Pushing data through the Provider's dataplane to the Consumer's sink

When the backend posts an event to the callback URL it received during subscription
(`{provider_url}/dataplane/proxy/{dataplaneId}`), the Provider's dataplane forwards it to the
Consumer's sink (`http://127.0.0.1:1112`) with the `Bearer TOKEN-ABCDEFG` header specified in the
Consumer's `dataAddress`.

In [5]:
provider_dp = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/transfer-process/{provider_transfer['id']}",
    headers=json_header,
))
provider_dp_id = provider_dp["id"]

push_res = show(requests.post(
    f"{data_space_provider}/dataplane/proxy/{provider_dp_id}",
    headers=json_header,
    json={"event_type": "hotel_waste_generated", "hotel": "Madrid Central", "kg": 18.5},
), ok=False)
if demo_sink_events:
    print("\nLast event received by Consumer sink (127.0.0.1:1112):")
    print(json.dumps(demo_sink_events[-1], indent=2))

HTTP 200 GET http://127.0.0.1:1200/api/v1/transfer-agent/dataplane/dataplane-processes/transfer-process/urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc
{
  "id": "urn:dataplane-transfer:e7e4234a-a529-4ed5-9f2f-6c66ad4ee090",
  "userId": "system",
  "userRole": "/admin",
  "visibility": "Public",
  "transferProcessId": "urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc",
  "role": "Provider",
  "interactionMode": "Push",
  "state": "Started",
  "connectorInstanceId": "urn:connector-instance:ef03612d-eb06-4f0e-9f16-c92f09296311",
  "ingressConfig": {
    "HttpListener": {
      "path": "/dataplane/proxy/urn:dataplane-transfer:e7e4234a-a529-4ed5-9f2f-6c66ad4ee090",
      "token": null,
      "token_type": null
    }
  },
  "egressConfig": {
    "HttpProxy": {
      "path": "http://127.0.0.1:1100/dataplane/proxy/urn:dataplane-transfer:7b1c909e-3f92-4676-a53b-eddb2537e542",
      "token": null,
      "token_type": null
    }
  },
  "flowControl": {
    "auth": {
      "typ

## 5. Suspend (`rpc/setup-suspension`, Provider -> Consumer)

The Provider pauses delivery; the subscription is dropped (`127.0.0.1:8082/subscriptions/unsubscribe/{id}`)
until the transfer starts again.

In [6]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferSuspensionMessage",
    "@id": transfer_provider_process_id,
    "code": "SUSPEND",
    "reason": ["Suspending for demonstration purposes"],
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-suspension", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "SUSPENDED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-suspension
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferSuspensionMessage",
    "code": "SUSPEND",
    "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
    "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb",
    "reason": [
      "Suspending for demonstration purposes"
    ]
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "SUSPENDED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": [
        "Suspending for demonstration purposes"
      ],
      "code": "SUSPEND"
    },
    "correlation": {
      "identifi

## 6. Resume (`rpc/setup-start`, Provider -> Consumer)

A suspended transfer resumes with another start message, re-subscribing to the backend.

In [7]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "@id": transfer_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "STARTED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-start
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferStartMessage",
    "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
    "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "STARTED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
      "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb",
      "agree

## 7. Role check: the Consumer cannot start

Only the Provider sends a `TransferStartMessage`. The Consumer's agent refuses the RPC before
anything goes on the wire and answers a `TransferError` naming the broken rule.

In [8]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferStartMessage",
    "@id": transfer_consumer_process_id,
}
response = requests.post(f"{data_space_consumer}/dsp/current/transfers/rpc/setup-start", headers=json_header, json=payload)
answer = show(response, ok=False)
assert response.status_code == 400, "the Consumer must not be able to start a transfer"

HTTP 400 POST http://127.0.0.1:1100/dsp/current/transfers/rpc/setup-start
{
  "@context": [
    "https://w3id.org/dspace/2025/1/context.jsonld"
  ],
  "@type": "TransferError",
  "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
  "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb",
  "code": "400",
  "reason": [
    "role: only a Provider sends a TransferStartMessage",
    "state: TransferStartMessage is not allowed from state STARTED"
  ]
}


## 8. Completion (`rpc/setup-completion`, Provider -> Consumer)

When everything is delivered the Provider completes the transfer. `COMPLETED` is terminal.

In [9]:
payload = {
    "@context": eunomia_context,
    "@type": "RpcTransferCompletionMessage",
    "@id": transfer_provider_process_id,
}
response = requests.post(f"{data_space_provider}/dsp/current/transfers/rpc/setup-completion", headers=json_header, json=payload)
provider_transfer = transfer_step(response, "COMPLETED")

HTTP 200 POST http://127.0.0.1:1200/dsp/current/transfers/rpc/setup-completion
{
  "@context": "https://w3id.org/eunomia/v1/context.jsonld",
  "@type": "RpcTransferResult",
  "request": {
    "@context": "https://w3id.org/eunomia/v1/context.jsonld",
    "@type": "RpcTransferCompletionMessage",
    "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
    "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb"
  },
  "response": null,
  "transferAgentModel": {
    "id": "urn:transfer-process:0a20597e-fb81-4ac8-8e2f-3df6ebe30adc",
    "userId": "system",
    "userRole": "/admin",
    "visibility": "Public",
    "role": "provider",
    "protocol": "dsp2025_1",
    "state": "COMPLETED",
    "stateMetadata": {
      "attribute": "BY_PROVIDER",
      "reason": null,
      "code": null
    },
    "correlation": {
      "identifiers": {},
      "consumerPid": "urn:uuid:7bb7b239-d597-4efe-90e4-cd10f70c9388",
      "providerPid": "urn:uuid:6027b644-d78f-4ddd-861a-18224e6c5bfb",


## 9. Final state, messages, dataplane logs and events (`/api/v1/transfer-agent/*`)

Each agent's Management API exposes its transfer process record (`/transfer-processes/{id}`),
exchanged DSP messages (`/transfer-messages/process/{id}`), dataplane transition logs
(`/dataplane/dataplane-processes/{dataplaneId}/logs`) and proxied data events
(`/dataplane/dataplane-processes/{dataplaneId}/events`).

In [10]:
for name, agent_url, process in [("Consumer", data_space_consumer, consumer_transfer), ("Provider", data_space_provider, provider_transfer)]:
    proc_view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-processes/{process['id']}", headers=json_header).json()
    msgs_view = requests.get(f"{agent_url}/api/v1/transfer-agent/transfer-messages/process/{process['id']}", headers=json_header).json()
    transitions = [f"{m.get('stateTransitionFrom') or 'Initial'}->{m['stateTransitionTo']}" for m in msgs_view["items"]]
    print(f"{name}: state={proc_view['state']}, messages={len(msgs_view['items'])} {transitions}")

dp_logs = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/{provider_dp_id}/logs",
    headers=json_header,
))
dp_events = show(requests.get(
    f"{data_space_provider}/api/v1/transfer-agent/dataplane/dataplane-processes/{provider_dp_id}/events",
    headers=json_header,
))

Consumer: state=COMPLETED, messages=5 ['STARTED->COMPLETED', 'SUSPENDED->STARTED', 'STARTED->SUSPENDED', 'REQUESTED->STARTED', 'INITIAL->REQUESTED']
Provider: state=COMPLETED, messages=5 ['STARTED->COMPLETED', 'SUSPENDED->STARTED', 'STARTED->SUSPENDED', 'REQUESTED->STARTED', 'INITIAL->REQUESTED']
HTTP 200 GET http://127.0.0.1:1200/api/v1/transfer-agent/dataplane/dataplane-processes/urn:dataplane-transfer:e7e4234a-a529-4ed5-9f2f-6c66ad4ee090/logs
[
  {
    "id": "urn:dataplane-process-log:48ee8db5-b1da-4881-b341-c7a0ab1ca133",
    "user_id": "system",
    "user_role": "/admin",
    "visibility": "Public",
    "dataplane_process_id": "urn:dataplane-transfer:e7e4234a-a529-4ed5-9f2f-6c66ad4ee090",
    "previous_state": null,
    "new_state": "Init",
    "trigger": "Creation",
    "reason": null,
    "created_at": "2026-10-17T13:58:37.224226Z"
  },
  {
    "id": "urn:dataplane-process-log:543a1959-fa45-4c70-9ff1-cdbbd14b2640",
    "user_id": "system",
    "user_role": "/admin",
    "visibil